# Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df_deep_train = pd.read_csv("/kaggle/input/multiclass-deep-features/train_features_multiclass.csv")
df_deep_test = pd.read_csv("/kaggle/input/multiclass-deep-features/test_features_multiclass.csv")
df_handcrafted_train = pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Hnadcrafted_Train_features_multiclass.csv")
df_handcrafted_test = pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Handcraftred_Test_features_multiclass.csv")


df_deep_train['id'] = df_deep_train['image_id']
df_deep_test['id'] = df_deep_test['image_id']

df_fusion_train = pd.merge(df_deep_train, df_handcrafted_train, on='id', how='inner')
df_fusion_test = pd.merge(df_deep_test, df_handcrafted_test, on='id', how='inner')


X_train_fusion = df_fusion_train.drop(columns=['image_id', 'id', 'label_y', 'label_x'])
X_test_fusion = df_fusion_test.drop(columns=['image_id', 'id', 'label_y', 'label_x'])
y_train = df_fusion_train['label_y'].values
y_test = df_fusion_test['label_y'].values

X_train_deep = df_deep_train.drop(columns=['image_id', 'id', 'label'])
X_test_deep = df_deep_test.drop(columns=['image_id', 'id', 'label'])
y_train_deep=df_deep_train['label']
y_test_deep=df_deep_test['label']

X_deep=pd.concat((df_deep_train,df_deep_test),axis=0)
y_deep=X_deep['label']
X_deep=X_deep.drop(columns=['image_id','id','label'])

X_train_hand = df_handcrafted_train.drop(columns=['id', 'label'])
X_test_hand = df_handcrafted_test.drop(columns=['id', 'label'])
y_train_hand=df_handcrafted_train['label']
y_test_hand=df_handcrafted_test['label']

X_handcrafted=pd.concat((df_handcrafted_train,df_handcrafted_test),axis=0)
y_handcrafted=X_handcrafted['label']
X_handcrafted=X_handcrafted.drop(columns=['id','label'])

X_total = np.concatenate((X_train_fusion, X_test_fusion), axis=0)
y_total = np.concatenate((y_train, y_test), axis=0)

In [ ]:
fix_map = {0: 3, 1: 2, 2: 1, 3: 0}

y_deep = y_deep.map(fix_map)
y_train_deep = y_train_deep.map(fix_map)
y_test_deep = y_test_deep.map(fix_map)

print("Deep labels remapped. Consistency check:")
print("Handcrafted Class 0 count:", np.sum(y_handcrafted == 0))
print("Deep Class 0 count:       ", np.sum(y_deep == 0)) 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score,precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.utils import resample

# ROC CURVE

In [ ]:
def compute_roc_bootstrap(y_true, y_scores, n_bootstraps=1000):

    tprs = []
    aucs = []
    mean_fpr = np.linspace(0, 1, 100)
    
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    base_auc = auc(fpr, tpr)
    
    rng = np.random.RandomState(42)
    for i in range(n_bootstraps):
        # Bootstrap sample indices
        indices = rng.randint(0, len(y_scores), len(y_scores))
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        y_true_boot = y_true[indices]
        y_scores_boot = y_scores[indices]
        
        fpr_boot, tpr_boot, _ = roc_curve(y_true_boot, y_scores_boot)
        
        interp_tpr = np.interp(mean_fpr, fpr_boot, tpr_boot)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(auc(fpr_boot, tpr_boot))
        
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_auc = np.std(aucs)
    
    tpr_lower = np.percentile(tprs, 2.5, axis=0)
    tpr_upper = np.percentile(tprs, 97.5, axis=0)
    auc_lower = np.percentile(aucs, 2.5)
    auc_upper = np.percentile(aucs, 97.5)
    
    return mean_fpr, mean_tpr, tpr_lower, tpr_upper, base_auc, auc_lower, auc_upper

def plot_roc_with_ci(model, X, y, class_names,model_name):

    # Sanitize inputs
    if hasattr(X, "values"): X = X.values
    if hasattr(y, "values"): y = y.values
    
    n_classes = len(np.unique(y))
    
    # Generate Probabilities via Cross-Validation (prevents leakage)
    if hasattr(model, "predict_proba"):
            print(f"Generating probabilities using trained {model_name}...")
            y_probas = model.predict_proba(X)
    else:
        print(f"Generating probabilities via CV for {model_name}...")
        y_probas = cross_val_predict(model, X, y, cv=5, method="predict_proba", n_jobs=-1)
    plt.figure(figsize=(10, 8))
    
    y_bin = label_binarize(y, classes=np.unique(y))
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    
    if class_names is None:
        class_names = [f"Class {i}" for i in range(n_classes)]

    for i, color in zip(range(n_classes), colors):
        print(f"Bootstrapping Class {class_names[i]}...")
        mean_fpr, mean_tpr, tpr_lower, tpr_upper, mean_auc, auc_low, auc_high =compute_roc_bootstrap(y_bin[:, i], y_probas[:, i])
        
        plt.plot(mean_fpr, mean_tpr, color=color, lw=2,label=f'{class_names[i]} (AUC = {mean_auc:.2f} [{auc_low:.2f}-{auc_high:.2f}])')
        plt.fill_between(mean_fpr, tpr_lower, tpr_upper, color=color, alpha=0.1)
        
    print("Bootstrapping Micro-Average...")
    mean_fpr, mean_tpr, tpr_lower, tpr_upper, mean_auc, auc_low, auc_high = \
        compute_roc_bootstrap(y_bin.ravel(), y_probas.ravel())
        
    plt.plot(mean_fpr, mean_tpr, color='deeppink', linestyle=':', lw=3,
             label=f'Micro-average (AUC = {mean_auc:.2f} [{auc_low:.2f}-{auc_high:.2f}])')

    
    all_fpr = np.unique(np.concatenate([np.linspace(0, 1, 100) for _ in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
         fpr_class, tpr_class, _ = roc_curve(y_bin[:, i], y_probas[:, i])
         mean_tpr += np.interp(all_fpr, fpr_class, tpr_class)
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    
    plt.plot(all_fpr, mean_tpr, color='navy', linestyle=':', lw=3,
             label=f'Macro-average (AUC = {macro_auc:.2f})')

    # Formatting
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curves with 95% Confidence Intervals\nModel: {model_name}', fontsize=14)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
pca_n_components=0.98

models= {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='linear', probability=True, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
}
model_PCA=Pipeline([
    ('scaler',StandardScaler()),
    ('PCA ',PCA(pca_n_components ,random_state=42)),
    ("clf",LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
    ])

In [ ]:
#for deep features include both logisitic regression and linear SVM
model=models['Logistic Regression'].fit(X_train_deep,y_train_deep)
plot_roc_with_ci( model,  X_test_deep, y_test_deep, class_names=['glioma', 'meningioma', 'notumor','pituitary'],model_name='Logistic Regression')

In [ ]:
#for fusion features include both logisitic regression and linear SVM
model=models['Logistic Regression'].fit(X_train_fusion,y_train)
plot_roc_with_ci( model,  X_test_fusion, y_test, class_names=['glioma', 'meningioma', 'notumor','pituitary'],model_name='Logistic Regression')

In [ ]:
#for handcrafted features include Random Forest
model=models['Random Forest'].fit(X_train_hand, y_train_hand)
plot_roc_with_ci( model,  X_test_hand ,y_test_hand, class_names=['glioma', 'meningioma', 'notumor','pituitary'],model_name='Random Forest')

In [ ]:
#for fusion+PCA features include logisitic regression
model=model_PCA.fit(X_train_fusion,y_train)
plot_roc_with_ci( model,  X_test_fusion, y_test, class_names=['glioma', 'meningioma', 'notumor','pituitary'],model_name='Logistic Regression')